# OAGH Benchmark Evaluator

This notebook automatically extracts `run_*` IDs from Slurm report files instead of hardcoding them.

**Workflow:**
1. Scan `Report-*.out` files in `Slurm_Training/` to extract `run_*` timestamps
2. Load trained model weights matching those run IDs
3. Evaluate all models across test datasets and SNR levels
4. Generate heatmaps, line plots, and box plots


In [ ]:
import os
import re
import glob

def extract_run_ids_from_reports(slurm_dir: str) -> dict:
    """
    Scan Report-*.out files in slurm_dir and extract the last run_YYYYMMDD_HHMMSS
    identifier from each file.
    
    Returns:
        dict mapping report filename -> run_id (or None if not found)
    """
    run_pattern = re.compile(r'(run_\d{8}_\d{6})')
    results = {}
    
    report_files = sorted(glob.glob(os.path.join(slurm_dir, 'Report-*.out')))
    
    for report_path in report_files:
        basename = os.path.basename(report_path)
        
        # Read the file and find all run_* matches, take the last one
        with open(report_path, 'r') as f:
            content = f.read()
        
        matches = run_pattern.findall(content)
        if matches:
            results[basename] = matches[-1]
        else:
            # No run_* found (e.g., direct inversion has no ML training)
            results[basename] = None
    
    return results


# ---- Extract run IDs ----
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))
SLURM_DIR = os.path.join(os.getcwd(), 'Slurm_Training')

report_to_run = extract_run_ids_from_reports(SLURM_DIR)

# Filter out None (direct inversion) and deduplicate
RUN_IDS = list(set(rid for rid in report_to_run.values() if rid is not None))
RUN_IDS.sort()  # deterministic order

print(f'Found {len(report_to_run)} report files in {SLURM_DIR}:')
print(f'{"":-<70}')
for report, run_id in sorted(report_to_run.items()):
    status = run_id if run_id else '(no ML run — direct inversion)'
    print(f'  {report:45s} -> {status}')
print(f'{"":-<70}')
print(f'\nUnique RUN_IDS ({len(RUN_IDS)}):')
for rid in RUN_IDS:
    print(f'  {rid}')


In [ ]:
import os
import re
import torch
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional
from Data_loader import get_dataloader_for_test
from train_functions import wave_number_to_shear_stiffness, setup_model
from evaluation.metrics import compute_mae, compute_rmse, compute_ssim, compute_cnr, get_masks
from datetime import datetime
from pathlib import Path
import json


class ModelEvaluator:
    """Comprehensive model evaluation across datasets and SNR levels."""
    
    def __init__(
        self,
        root_dir: str = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Model_Logs",
        device: str = "cuda"
    ):
        self.root = Path(root_dir)
        self.device = device
        
    def find_weights_by_runs(self, run_ids: List[str]) -> List[str]:
        """
        Find all weights.pth matching specific run IDs.
        
        Args:
            run_ids: List of run IDs like ["run_20260223_133208", "run_20260223_140110"]
        """
        # Extract just the numeric parts if full run_YYYYMMDD_HHMMSS provided
        numeric_ids = []
        for rid in run_ids:
            if rid.startswith("run_"):
                # Extract YYYYMMDD_HHMMSS part
                parts = rid.split("_")
                if len(parts) >= 3:
                    numeric_ids.append(parts[-1])  # Get HHMMSS
                else:
                    numeric_ids.append(rid)
            else:
                numeric_ids.append(rid)
        
        # Build regex pattern
        run_pat = re.compile(r"run_\d{8}_(" + "|".join(numeric_ids) + r")")
        
        def bs64_has_matching_tb(bs64_dir: Path) -> bool:
            """Check if bs64 directory contains matching tensorboard logs."""
            # Fast path: look for tfevents files with matching run id
            for p in bs64_dir.rglob("*"):
                s = str(p)
                if run_pat.search(s) and ("tfevents" in p.name or "events.out.tfevents" in p.name):
                    return True
            
            # Fallback: any path under bs64 contains the run id
            for p in bs64_dir.rglob("*"):
                if run_pat.search(str(p)):
                    return True
            
            return False
        
        matched_weights = []
        
        # Find all weights.pth
        for w in self.root.rglob("weights.pth"):
            bs64_dir = w.parent  # .../bs64
            if bs64_dir.name != "bs64":
                # If weights live deeper, you can remove this guard
                continue
            
            if bs64_has_matching_tb(bs64_dir):
                matched_weights.append(str(w))
        
        print(f"Matched weights.pth files: {len(matched_weights)}")
        for p in matched_weights:
            print(f"  {p}")
        
        return matched_weights
    
    def find_weights_by_filter(self, filter_dict: Dict[str, any]) -> List[str]:
        """
        Find weights.pth based on flexible filters.
        
        filter_dict examples:
            {"loss_type": "mse"}
            {"loss_type": "residual", "lambda_physics": 0.025}
            {"loss_type": ["mse", "residual"]}
        """
        matched_weights = []
        
        for w in self.root.rglob("weights.pth"):
            # Load checkpoint to check metadata
            try:
                ckpt = torch.load(w, map_location="cpu")
                
                # Check all filter conditions
                match = True
                for key, value in filter_dict.items():
                    if key not in ckpt:
                        match = False
                        break
                    
                    # Handle list of acceptable values
                    if isinstance(value, list):
                        if ckpt[key] not in value:
                            match = False
                            break
                    else:
                        if ckpt[key] != value:
                            match = False
                            break
                
                if match:
                    matched_weights.append(str(w))
                    
            except Exception as e:
                print(f"Warning: Could not load {w}: {e}")
                continue
        
        print(f"Found {len(matched_weights)} weights matching filter: {filter_dict}")
        return matched_weights
    
    def load_models(self, weight_paths: List[str]) -> Dict[str, torch.nn.Module]:
        """Load models from weight paths."""
        models = {}
        
        for pth_path in weight_paths:
            # Create key based on relative path from root
            path_obj = Path(pth_path)
            key = os.path.relpath(os.path.dirname(pth_path), self.root)
            
            # Load checkpoint
            ckpt = torch.load(pth_path, map_location=self.device)
            
            # Setup model
            model = setup_model("FDTDNet")[0]
            model.load_state_dict(ckpt["state_dict"])
            model.eval()
            model.to(self.device)
            
            models[key] = model
        
        print(f"\nLoaded {len(models)} models:")
        for k in models.keys():
            print(f"  - {k}")
        
        return models
    
    def evaluate_per_sample_metrics(
        self,
        model: torch.nn.Module,
        test_loader,
        has_inclusions: bool = False
    ) -> pd.DataFrame:
        """Evaluate model per sample and return DataFrame."""
        model.eval()
        records = []
        
        with torch.no_grad():
            for wave, mu, k_gt, mfre, fov, index in test_loader:
                wave = wave.to(self.device)
                mu = mu.to(self.device)
                k_gt = k_gt.to(self.device)
                mfre = mfre.to(self.device, dtype=wave.dtype)
                fov = fov.to(self.device, dtype=wave.dtype)
                
                # Permute wave
                wave = wave.permute(4, 0, 1, 2, 3).permute(1, 2, 0, 3, 4)
                
                # Predict
                pred_k = model(wave)
                pred_mu = wave_number_to_shear_stiffness(pred_k, mfre, fov)
                
                # Metrics per sample
                batch_size = wave.shape[0]
                for b in range(batch_size):
                    record = {
                        "index": int(index[b]),
                        "mae_k": compute_mae(pred_k[b], k_gt[b]),
                        "rmse_k": compute_rmse(pred_k[b], k_gt[b]),
                        "ssim_k": compute_ssim(pred_k[b], k_gt[b]),
                        "mae_mu": compute_mae(pred_mu[b], mu[b]),
                        "rmse_mu": compute_rmse(pred_mu[b], mu[b]),
                        "ssim_mu": compute_ssim(pred_mu[b], mu[b])
                    }
                    
                    # CNR if inclusions
                    if has_inclusions:
                        inc_mask, bg_mask = get_masks(mu[b])
                        cnr_value = compute_cnr(
                        pred_mu[b].squeeze(),
                        inc_mask,
                        bg_mask
                        )
                        record["cnr"] = cnr_value.item() if torch.is_tensor(cnr_value) else cnr_value
                    records.append(record)  # ← This line was missing!

        
        return pd.DataFrame(records)
    
    def evaluate_aggregate_metrics(
        self,
        model: torch.nn.Module,
        test_loader,
        has_inclusions: bool = False
    ) -> Dict:
        """Evaluate model and return aggregated metrics."""
        model.eval()
        metrics = {
            "mae_k": [], "rmse_k": [], "ssim_k": [],
            "mae_mu": [], "rmse_mu": [], "ssim_mu": []
        }
        if has_inclusions:
            metrics["cnr"] = []
        
        with torch.no_grad():
            for wave, mu, k_gt, mfre, fov, index in test_loader:
                wave = wave.to(self.device)
                mu = mu.to(self.device)
                k_gt = k_gt.to(self.device)
                mfre = mfre.to(self.device, dtype=wave.dtype)
                fov = fov.to(self.device, dtype=wave.dtype)
                
                wave = wave.permute(4, 0, 1, 2, 3).permute(1, 2, 0, 3, 4)
                
                pred_k = model(wave)
                pred_mu = wave_number_to_shear_stiffness(pred_k, mfre, fov)
                
                metrics["mae_k"].append(compute_mae(pred_k, k_gt))
                metrics["rmse_k"].append(compute_rmse(pred_k, k_gt))
                metrics["ssim_k"].append(compute_ssim(pred_k, k_gt))
                metrics["mae_mu"].append(compute_mae(pred_mu, mu))
                metrics["rmse_mu"].append(compute_rmse(pred_mu, mu))
                metrics["ssim_mu"].append(compute_ssim(pred_mu, mu))
                
                if has_inclusions:
                    inc_mask, bg_mask = get_masks(mu)
                    metrics["cnr"].append(compute_cnr(pred_mu.squeeze(), inc_mask, bg_mask))
        
        return {k: (torch.tensor(v).mean().item() if v else 0.0) for k, v in metrics.items()}
    
    def run_full_evaluation(
        self,
        models: Dict[str, torch.nn.Module],
        test_configs: List[Dict],
        per_sample: bool = False
    ) -> pd.DataFrame:
        """
        Run evaluation across models and test configurations.
        
        test_configs: List of dicts with keys:
            - test_dir: path to test data
            - snr_levels: list of SNR values (None for clean)
            - has_inclusions: bool
            - dataset_name: str (for labeling)
        """
        all_rows = []
        
        for config in test_configs:
            test_dir = config["test_dir"]
            snr_levels = config["snr_levels"]
            has_inclusions = config.get("has_inclusions", False)
            dataset_name = config.get("dataset_name", "Unknown")
            
            print(f"\n{'='*60}")
            print(f"Evaluating on: {dataset_name}")
            print(f"{'='*60}")
            
            for model_name, model in models.items():
                for snr in snr_levels:
                    loader = get_dataloader_for_test(
                        test_dir,
                        batch_size=1,
                        snr_db=snr
                    )
                    
                    if per_sample:
                        df = self.evaluate_per_sample_metrics(
                            model, loader, has_inclusions
                        )
                        df["model"] = model_name
                        df["snr_db"] = snr if snr is not None else "clean"
                        df["dataset"] = dataset_name
                        all_rows.append(df)
                    else:
                        metrics = self.evaluate_aggregate_metrics(
                            model, loader, has_inclusions
                        )
                        row = {
                            "model": model_name,
                            "snr_db": snr if snr is not None else "clean",
                            "dataset": dataset_name,
                            **metrics
                        }
                        all_rows.append(pd.DataFrame([row]))
                
                print(f"  ✓ Completed {model_name}")
        
        return pd.concat(all_rows, ignore_index=True)


# ==================== USAGE EXAMPLE ====================



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def main():
    # Initialize evaluator
    evaluator = ModelEvaluator(device="cuda")
    
    # Use dynamically extracted RUN_IDS (from cell above)
    print(f"Using {len(RUN_IDS)} run IDs extracted from Slurm report files")
    weight_paths = evaluator.find_weights_by_runs(RUN_IDS)
    
    # Load models
    models = evaluator.load_models(weight_paths)
    
    # Define test configurations
    base_test_dir = "/storage/home/hcoda1/2/jueda3/r-jueda3-0/MRE_OAGH_test/MinimalCode_3_5_2026/Data"
    snr_levels = [30, 25, 20, 15]
    
    test_configs = [
        # Main test set
        {
            "test_dir": f"{base_test_dir}/Test/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": False,
            "dataset_name": "Main_Test"
        },
        # Inclusion datasets
        {
            "test_dir": f"{base_test_dir}/Inclusion/R0_3125cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R0.3125cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R0_625cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R0.625cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R1_25cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R1.25cm"
        },
        {
            "test_dir": f"{base_test_dir}/Inclusion/R2_5cm/pt_files/",
            "snr_levels": snr_levels,
            "has_inclusions": True,
            "dataset_name": "Inclusion_R2.5cm"
        }
    ]
    
    # Run evaluations
    print("\n" + "="*60)
    print("AGGREGATE METRICS EVALUATION")
    print("="*60)
    df_aggregate = evaluator.run_full_evaluation(
        models, test_configs, per_sample=False
    )
    
    print("\n" + "="*60)
    print("PER-SAMPLE METRICS EVALUATION")
    print("="*60)
    df_per_sample = evaluator.run_full_evaluation(
        models, test_configs, per_sample=True
    )
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path("evaluation_results") / timestamp
    output_dir.mkdir(parents=True, exist_ok=True)
    plots_dir = output_dir / "plots"
    plots_dir.mkdir(exist_ok=True)

    # Save CSVs
    df_aggregate.to_csv(output_dir / "aggregate_metrics.csv", index=False)
    df_per_sample.to_csv(output_dir / "per_sample_metrics.csv", index=False)

    # Save metadata (includes which reports were used)
    metadata = {
        "timestamp": timestamp,
        "run_ids": RUN_IDS,
        "report_to_run_mapping": {k: v for k, v in report_to_run.items()},
        "num_models": len(models),
        "datasets": df_aggregate["dataset"].unique().tolist(),
        "snr_levels": df_aggregate["snr_db"].unique().tolist(),
        "total_samples_evaluated": len(df_per_sample),
        "models_evaluated": list(models.keys())
    }

    with open(output_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    return df_aggregate, df_per_sample, output_dir, plots_dir



In [ ]:
df_aggregate, df_per_sample, output_dir, plots_dir = main()


In [ ]:
# Generate and save plots for each dataset
print("Generating plots...")
datasets = df_aggregate["dataset"].unique()

for dataset in datasets:
    print(f"  Plotting {dataset}...")
    dataset_safe = dataset.replace("/", "_").replace(" ", "_").replace('.','_')

    # Determine metrics (add CNR if inclusions)
    metrics = ["mae", "rmse"]
    has_cnr = "Inclusion" in dataset and "cnr" in df_aggregate.columns
    if has_cnr:
        metrics.append("cnr")

    # ========== COMBINED HEATMAP FIGURE ==========
    n_metrics = len(metrics)
    fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
    if n_metrics == 1:
        axes = axes.reshape(1, -1)

    for idx, metric in enumerate(metrics):
        plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()

        has_k = f"{metric}_k" in plot_df.columns
        has_mu = f"{metric}_mu" in plot_df.columns

        if has_k and has_mu:
            pivot_k = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_k")
            sns.heatmap(pivot_k, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="magma")
            axes[idx, 0].set_title(f"(k) {metric.upper()}")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel("Model")

            pivot_mu = plot_df.pivot(index="model", columns="snr_db", values=f"{metric}_mu")
            sns.heatmap(pivot_mu, annot=True, fmt=".2f", ax=axes[idx, 1], cmap="magma")
            axes[idx, 1].set_title(f"(mu) {metric.upper()}")
            axes[idx, 1].set_xlabel("SNR (dB)")
            axes[idx, 1].set_ylabel("")
        else:
            pivot = plot_df.pivot(index="model", columns="snr_db", values=metric)
            sns.heatmap(pivot, annot=True, fmt=".2f", ax=axes[idx, 0], cmap="viridis")
            axes[idx, 0].set_title(f"{metric.upper()}")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel("Model")
            axes[idx, 1].axis('off')

    plt.suptitle(f"{dataset} - Heatmaps", fontsize=16, y=1.002)
    plt.tight_layout()
    plt.savefig(plots_dir / f"{dataset_safe}_all_heatmaps.png", dpi=300, bbox_inches='tight')
    plt.close()

    # ========== COMBINED LINE PLOT FIGURE ==========
    fig, axes = plt.subplots(n_metrics, 2, figsize=(16, 5*n_metrics))
    if n_metrics == 1:
        axes = axes.reshape(1, -1)

    plot_df = df_aggregate[df_aggregate["dataset"] == dataset].copy()
    plot_df["snr_numeric"] = plot_df["snr_db"].apply(lambda x: 999 if x == "clean" else float(x))
    plot_df["snr_label"] = plot_df["snr_db"].apply(lambda x: "Clean" if x == "clean" else str(x))
    plot_df = plot_df.sort_values("snr_numeric")

    for idx, metric in enumerate(metrics):
        has_k = f"{metric}_k" in plot_df.columns
        has_mu = f"{metric}_mu" in plot_df.columns

        if has_k and has_mu:
            sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_k", 
                        hue="model", marker="o", style="model", ax=axes[idx, 0])
            unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
            axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
            axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
            axes[idx, 0].set_title(f"(k) {metric.upper()} vs SNR")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel(metric.upper())
            axes[idx, 0].grid(True)
            axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

            sns.lineplot(data=plot_df, x="snr_numeric", y=f"{metric}_mu",
                        hue="model", marker="o", style="model", ax=axes[idx, 1])
            axes[idx, 1].set_xticks(unique_snr["snr_numeric"])
            axes[idx, 1].set_xticklabels(unique_snr["snr_label"])
            axes[idx, 1].set_title(f"(mu) {metric.upper()} vs SNR")
            axes[idx, 1].set_xlabel("SNR (dB)")
            axes[idx, 1].set_ylabel(metric.upper())
            axes[idx, 1].grid(True)
            axes[idx, 1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        else:
            sns.lineplot(data=plot_df, x="snr_numeric", y=metric,
                        hue="model", marker="o", style="model", ax=axes[idx, 0])
            unique_snr = plot_df.sort_values("snr_numeric")[["snr_numeric", "snr_label"]].drop_duplicates()
            axes[idx, 0].set_xticks(unique_snr["snr_numeric"])
            axes[idx, 0].set_xticklabels(unique_snr["snr_label"])
            axes[idx, 0].set_title(f"{metric.upper()} vs SNR")
            axes[idx, 0].set_xlabel("SNR (dB)")
            axes[idx, 0].set_ylabel(metric.upper())
            axes[idx, 0].grid(True)
            axes[idx, 0].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            axes[idx, 1].axis('off')

    plt.suptitle(f"{dataset} - Metrics vs SNR", fontsize=16, y=1.002)
    plt.tight_layout()
    plt.savefig(plots_dir / f"{dataset_safe}_all_lineplot.png", dpi=300, bbox_inches='tight')
    plt.close()

    # ========== COMBINED BOX PLOT FIGURE (PER-SAMPLE) ==========
    print(f"    Generating per-sample distributions...")
    box_metrics = ["mae_k", "rmse_k", "mae_mu", "rmse_mu", "ssim_k", "ssim_mu"]
    if has_cnr:
        box_metrics.append("cnr")

    available_metrics = [m for m in box_metrics if m in df_per_sample.columns]

    if available_metrics:
        n_box = len(available_metrics)
        fig, axes = plt.subplots(n_box, 1, figsize=(14, 4*n_box))
        if n_box == 1:
            axes = [axes]

        plot_df_sample = df_per_sample[df_per_sample["dataset"] == dataset].copy()

        for idx, metric in enumerate(available_metrics):
            sns.boxplot(data=plot_df_sample, x="snr_db", y=metric, 
                       hue="model", palette="Set2", ax=axes[idx])
            axes[idx].set_title(f"Per-sample {metric} distribution")
            axes[idx].set_xlabel("SNR (dB)")
            axes[idx].set_ylabel(metric)
            axes[idx].legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

        plt.suptitle(f"{dataset} - Per-Sample Distributions", fontsize=16, y=1.002)
        plt.tight_layout()
        plt.savefig(plots_dir / f"{dataset_safe}_all_boxplots.png", dpi=300, bbox_inches='tight')
        plt.close()

print("All plots saved!")
print(f"\nResults saved to: {output_dir}/")
print(f"  aggregate_metrics.csv")
print(f"  per_sample_metrics.csv")
print(f"  metadata.json")
print(f"  plots/ ({len(list(plots_dir.glob('*.png')))} images)")

